# Story Context Selection and Enrichment Benchmark

This notebook isolates context-composition bottlenecks after the token-budget benchmark showed that increasing the Core budget from 6000 to 32000 did not remove truncation. It must be run separately from `story-context-budget-benchmark.ipynb`.

The experiment does **not** turn DevLog into a retrieval service. It measures bounded Core composition policies: candidate selection, content enrichment, document budgets, symbol budgets, and the final Story Context envelope.

## Hypothesis

The observed `truncated=true` at 32000 Core tokens is caused by one or more non-token ceilings:

- the maximum number of selected evidence items;
- file-content enrichment limits;
- document-content limits;
- symbol inspection limits;
- the Story Context serialized envelope (`max-bytes` / `max-estimated-tokens`);
- missing document candidates or Story-aware selection behavior.

The experiment must identify which warning and measured field changes when exactly one ceiling changes.

## Fixed conditions

The following conditions remain constant for every profile:

- Core token budget: `32000`;
- provider/model and output policy;
- repository revision;
- intent and semantic question;
- scenario files and guidance;
- three repetitions per scenario;
- unique run nonce, recorded in the capture;
- authenticated snapshot retrieval after every execution;
- no Python-side retrieval or context expansion.

If the effective Core token budget is not `32000`, the run is invalid and must not be compared.

## Profiles

Each profile changes one policy family relative to `C0`, except `C5`, which tests the combined upper experiment. The values are experimental ceilings, not production recommendations.

| Profile | Changed variables | Purpose |
|---|---|---|
| `C0-current` | Current defaults | Control profile |
| `C1-selection` | `MAX_EVIDENCE_ITEMS=120` | Test the 60-item ceiling |
| `C2-file-content` | `MAX_FILES=12`, per-file `8000`, total `24000` | Test file enrichment ceilings |
| `C3-documents` | selected `10`, per-document `8000`, total `24000` | Test Story/ADR document ceilings |
| `C4-envelope` | agent context bytes `131072`, estimated tokens `32768` | Test final projection envelope |
| `C5-combined` | C1 + C2 + C3 + C4 | Test combined non-token ceilings |

All unlisted variables remain at their current values. Every capture must include the complete effective environment profile.

In [ ]:
PROFILES = {
    'C0-current': {},
    'C1-selection': {
        'REPOSITORY_CONTEXT_MAX_EVIDENCE_ITEMS': 120,
    },
    'C2-file-content': {
        'REPOSITORY_CONTEXT_CONTENT_MAX_FILES': 12,
        'REPOSITORY_CONTEXT_CONTENT_MAX_CHARACTERS_PER_FILE': 8000,
        'REPOSITORY_CONTEXT_CONTENT_MAX_TOTAL_CHARACTERS': 24000,
    },
    'C3-documents': {
        'REPOSITORY_CONTEXT_DOCUMENT_MAX_SELECTED': 10,
        'REPOSITORY_CONTEXT_DOCUMENT_MAX_CHARS': 8000,
        'REPOSITORY_CONTEXT_DOCUMENT_MAX_TOTAL_CHARS': 24000,
    },
    'C4-envelope': {
        'ENGINEERING_STORY_AGENT_CONTEXT_MAX_BYTES': 131072,
        'ENGINEERING_STORY_AGENT_CONTEXT_MAX_ESTIMATED_TOKENS': 32768,
    },
}
PROFILES['C5-combined'] = {
    key: value for name in ('C1-selection', 'C2-file-content', 'C3-documents', 'C4-envelope')
    for key, value in PROFILES[name].items()
}
PROFILES

## Scenario corpus

Use the same four scenarios as the token benchmark, plus one multi-document scenario:

- broad Story Context question with Story and Compose files;
- targeted runtime component question;
- insufficient-evidence question;
- deliberately truncated-context question;
- multi-source question requesting Story 0112, ADR-067, ADR-079, `docker-compose.yml`, and `SelectedFileContentEnricher.java`.

The multi-source scenario is the primary diagnostic case. It must report each requested file as one of:

- absent from candidates;
- selected without content;
- selected with truncated content;
- selected with complete content.

## Runtime procedure

For each profile, start the backend with the fixed token budget and profile variables. The command must be captured in the result manifest. Example:

```bash
REPOSITORY_CONTEXT_MAX_TOKENS=32000 \
REPOSITORY_CONTEXT_MAX_EVIDENCE_ITEMS=120 \
docker compose up -d backend
```

Wait for backend readiness before the first request. Execute the robust runner and retrieve the authenticated snapshot for every task. Reject captures when the snapshot budget, profile, analysis, or repository revision does not match the expected run. Restore the baseline environment after the experiment.

## Required measurements

Capture these fields per run:

- effective profile variables and Core token budget;
- candidate, selected, discarded, and used-token counts;
- every accounting warning code;
- selected item count by layer and kind;
- requested-file coverage and content status;
- document count and document content status;
- serialized projection bytes and estimated tokens;
- Story Context envelope bytes and estimated tokens;
- provider input estimate and output allowance;
- context and projection digests;
- completion status, timeout, latency, and analysis availability;
- confidence, missing-information count, and implementation-question count.

The benchmark must distinguish `CONTENT_ENRICHMENT_TRUNCATED`, `EVIDENCE_SUMMARY_TRUNCATED`, `REPOSITORY_CONTEXT_BUDGET_APPLIED`, and final envelope reduction. A generic `truncated=true` is not enough to identify the bottleneck.

## Interpretation rules

- If C1 changes selection counts or removes budget warnings, the item ceiling is a bottleneck.
- If C2 changes requested-file content status or content warnings, file enrichment is a bottleneck.
- If C3 adds Story/ADR content, document budgeting or document candidate handling is a bottleneck.
- If C4 changes serialized projection size or final truncation while earlier profiles do not, the Story Context envelope is a bottleneck.
- If C5 is the only profile that improves coverage, the system has interacting ceilings; do not attribute the gain to the token budget alone.
- If no profile changes the warnings or content statuses, the bottleneck is likely candidate collection, Story-aware filtering, or cache/snapshot identity.
- A larger profile is not considered better if it increases timeout rate or cost without improving grounded usefulness.

No production policy is selected from this notebook alone.

## Decision record

Complete after execution:

- Primary bottleneck: _pending_
- Profile with measurable improvement: _pending_
- Grounded usefulness change: _pending_
- Timeout/latency change: _pending_
- Cost change: _pending_
- Production change authorized: _pending human decision_

The result must state whether the issue is a token ceiling, a selection/enrichment ceiling, a candidate-coverage issue, an envelope issue, or a combination.